# File A: Confidence Score Pipeline

Ministral annotates with confidence 1-5.
Rows with confidence <= 3 are rerouted to Llama 70B.
Final label comes from whichever model handled the row.
Then we evaluate against gold standard and compare with File B.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import time
import os
import krippendorff
from dotenv import load_dotenv
from sklearn.metrics import (
    accuracy_score, classification_report,
    f1_score, cohen_kappa_score, confusion_matrix
)
import importlib
import annotation_setup
importlib.reload(annotation_setup)
from annotation_setup import VALID_LABELS, instructions, reminder, parse_label, parse_explanation

load_dotenv()
api_key = os.getenv("CAC_API_KEY")

HOST          = "https://maki.uni-mannheim.de/v1"
MODEL_SMALL   = "ministral-3-14b"
MODEL_LARGE   = "Meta-Llama-3.3-70B-Instruct-AWQ-INT4"
CONFIDENCE_THRESHOLD = 3

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Small model:  {MODEL_SMALL}")
print(f"Large model:  {MODEL_LARGE}")
print(f"Routing threshold: confidence <= {CONFIDENCE_THRESHOLD} -> {MODEL_LARGE}")

Small model:  ministral-3-14b
Large model:  Meta-Llama-3.3-70B-Instruct-AWQ-INT4
Routing threshold: confidence <= 3 → Meta-Llama-3.3-70B-Instruct-AWQ-INT4


In [2]:
# load few-shot examples from step 4
# paste your current FEW_SHOT_EXAMPLES list here
FEW_SHOT_EXAMPLES = [
    # 1. Clear NO_CRIME_FRAME, integration context, no security link
    {
        "text": "Die Lage in der Unterkunft hat sich nach Angaben der Sozialarbeiter entspannt. Viele [Gruppe 1] haetten inzwischen Deutschkurse begonnen und bemuehten sich aktiv um eine Arbeitsstelle. Die Caritas lobte das Engagement der Freiwilligen, die regelmaessig Unterstuetzung anbieten.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Der Absatz beschreibt Integration und ehrenamtliche Hilfe ohne jeden Bezug zu Kriminalitaet, Sicherheit oder Praevention."
    },
    # 2. Clear CRIME_FRAME_NEG, direct perpetrator
    {
        "text": "Ein 34-jaehriger [Gruppe 1] wurde am Freitagabend am Hauptbahnhof festgenommen. Er soll in den vergangenen Wochen mehrfach Passanten mit einem Messer bedroht und Handtaschen entrissen haben. Die Polizei hat ihn als Serientaeter eingestuft.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Ein Gruppenmitglied wird explizit als Taeter schwerer Straftaten dargestellt und direkt mit Kriminalitaet verknuepft."
    },
    # 3. CRIME_FRAME_NEG, institutional security framing (fixes missed cases)
    {
        "text": "Die Innenministerkonferenz hat beschlossen, die erkennungsdienstliche Erfassung aller einreisenden [Gruppe 1] zu verschaerfen. Innenminister Hoffmann erklaerte, man koenne sich keine Sicherheitsluecken leisten. Es muesse sichergestellt werden, dass keine Schleuser, Kriminellen oder Extremisten die Situation ausnutzten.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Auch ohne konkrete Straftat wird die Gruppe institutionell als potenzielle Sicherheitsgefahr gerahmt — die explizite Verknuepfung mit Kriminalitaet und Extremismus durch eine Regierungsstimme ist CRIME_FRAME_NEG."
    },
    # 4. NO_CRIME_FRAME, group is victim of violence
    {
        "text": "Auf dem Gelaende der Fluechtlingsunterkunft in Tempelhof wurden in der Nacht Fensterscheiben eingeworfen und Hakenkreuze gesprueht. Die Polizei ermittelt wegen schwerer Sachbeschaedigung und Volksverhetzung. Die [Gruppe 1] Bewohner wurden unverletzt, aber stark veraengstigt in eine Notunterkunft gebracht.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Die [Gruppe 1] sind Opfer eines rechtsextremen Angriffs — eine Gruppe als Opfer von Gewalt darzustellen ist kein Sicherheitsrahmen gegen diese Gruppe."
    },
    # 5. NO_CRIME_FRAME, international war reporting between states
    {
        "text": "Die [Gruppe 1] Streitkraefte haben nach Angaben des Verteidigungsministeriums in Kiew erneut mehrere Staedte im Osten beschossen. Bei den Angriffen kamen nach ukrainischen Angaben mindestens zwoelf Zivilisten ums Leben. Die NATO beriet ueber weitere Waffenlieferungen.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Kriegsberichterstattung ueber einen Konflikt zwischen Staaten ohne Bezug zu Minderheiten in Deutschland erhaelt immer NO_CRIME_FRAME."
    },
    # 6. CRIME_FRAME_NEG, group legitimizes terrorism without being direct perpetrator
    {
        "text": "Sprecher der [Gruppe 1] bezeichneten den Anschlag als legitimen Widerstand und erklaerten, die Opfer seien selbst schuld. In sozialen Medien verbreiteten Anhaenger der Gruppe Videos, in denen der Terrorakt gefeiert wurde.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe legitimiert und feiert explizit einen Terroranschlag — das zaehlt als CRIME_FRAME_NEG auch ohne direkte Taeterschaft."
    },
    # 7. CRIME_FRAME_POS, successful security operation
    {
        "text": "Das Gemeinsame Terrorismusabwehrzentrum hat nach intensiver Zusammenarbeit von Polizei und Geheimdiensten einen islamistischen Anschlag in Muenchen vereitelt. Drei Verdaechtige wurden festgenommen. Innenministerin Weber sprach von einem grossen Erfolg der Sicherheitsbehoerden.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Die Sicherheitsbehoerden werden explizit als erfolgreiche Akteure gerahmt die eine Bedrohung abgewendet haben — Schutz und Sicherheitsgewinn stehen im Vordergrund."
    },
    # 8. CRIME_FRAME_POS, social factors explanation + differentiation (aligns with Robin's definition)
    {
        "text": "Experten erklaeren den Anstieg der Jugendkriminalitaet in Brennpunktvierteln vor allem mit sozialer Ausgrenzung und fehlenden Perspektiven fuer [Gruppe 1] Jugendliche. Kriminologin Mueller betonte, man duerfe keinen Generalverdacht gegen die Gruppe hegen — die grosse Mehrheit halte sich an Gesetze.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Kriminalitaet wird auf soziale Ursachen zurueckgefuehrt und eine explizite Differenzierung wird vorgenommen — das ist CRIME_FRAME_POS nach der Praventions- und Differenzierungsdefinition."
    },
    # 9. CRIME_FRAME_POS, prevention/deradicalization program
    {
        "text": "Das neue Deradikalisierungsprogramm der Bundesregierung soll vor allem [Gruppe 1] Jugendliche ansprechen, die gefaehrdet sind, in extremistische Kreise abzudriften. Sozialarbeiter begleiten die Teilnehmer ueber zwei Jahre und helfen beim Einstieg in Ausbildung und Arbeit.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Ein explizites Praeventionsprogramm im Sicherheitskontext — Schutz und Praevention stehen im Vordergrund, nicht Taeterschaft."
    },
    # 10. NO_CRIME_FRAME, integration without security frame (boundary with CRIME_FRAME_POS)
    {
        "text": "Die Stadtratsfraktion diskutierte Massnahmen zur besseren Integration von [Gruppe 1]. Themen waren Sprachkurse, Wohnraumvermittlung und Beschaeftigungsfoerderung. Ein konkreter Sicherheits- oder Kriminalitaetsbezug wurde nicht hergestellt.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Integration ohne jeden Sicherheits- oder Kriminalitaetsbezug ist NO_CRIME_FRAME — CRIME_FRAME_POS erfordert einen expliziten Sicherheits- oder Praeventionsrahmen."
    },
    # fixes id 14, 21, 135, generalised threat/disorder framing
    {
        "text": "Die Behörden berichten, dass sich in Teilen der Stadt regelrechte Parallelgesellschaften gebildet haben, in denen [Gruppe 1] unter sich bleiben und staatliche Kontrolle kaum noch greift. Anwohner klagen über Verwahrlosung und mangelnde Sicherheit.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe wird mit Kontrollverlust, Unordnung und mangelnder Sicherheit im deutschen Inland verknüpft — das ist generalisierte gesellschaftliche Bedrohungsrahmung und zählt als CRIME_FRAME_NEG."
    },

    # fixes id 30, explicit domestic security risk framing
    {
        "text": "Der Botschafter warnte: 'Diese [Gruppe 1] stellen ein Sicherheitsrisiko für Deutschland dar.' Es sei falsch, sie aufzunehmen, da sie in deutschen Städten für Unruhe sorgen würden.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe wird von einer öffentlichen Person explizit als Sicherheitsrisiko für Deutschland bezeichnet — das ist CRIME_FRAME_NEG unabhängig davon ob ein konkretes Verbrechen genannt wird."
    },
    # fixes id 15, 47, 113, 123, 189, political commentary overcalling
    {
        "text": "Die Politikerin forderte eine bessere Integrationspolitik für [Gruppe 1]. In der Debatte wurden auch Fragen der öffentlichen Ordnung angesprochen, ohne jedoch konkrete Vorfälle oder Straftaten zu nennen.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Allgemeine politische Debatten über Integration oder öffentliche Ordnung ohne explizite Verknüpfung der Gruppe mit konkreten Straftaten, Verdacht oder Ermittlungen sind NO_CRIME_FRAME."
    }
]

print(f"Few-shot examples loaded: {len(FEW_SHOT_EXAMPLES)}")
for i, ex in enumerate(FEW_SHOT_EXAMPLES):
    print(f"  {i+1}. {ex['label']}")

Few-shot examples loaded: 13
  1. NO_CRIME_FRAME
  2. CRIME_FRAME_NEG
  3. CRIME_FRAME_NEG
  4. NO_CRIME_FRAME
  5. NO_CRIME_FRAME
  6. CRIME_FRAME_NEG
  7. CRIME_FRAME_POS
  8. CRIME_FRAME_POS
  9. CRIME_FRAME_POS
  10. NO_CRIME_FRAME
  11. CRIME_FRAME_NEG
  12. CRIME_FRAME_NEG
  13. NO_CRIME_FRAME


In [3]:
def call_api(messages, model, temperature=0.0):
    payload = {
        "model": model,
        "temperature": temperature,
        "messages": messages
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    for attempt in range(3):
        try:
            r = requests.post(
                f"{HOST}/chat/completions",
                json=payload, headers=headers, timeout=120
            )
            if r.status_code == 200:
                return r.json()["choices"][0]["message"]["content"].strip()
            print(f"  [attempt {attempt+1}] status {r.status_code}")
        except Exception as e:
            print(f"  [attempt {attempt+1}] error: {e}")
            time.sleep(5)
    return "ERROR"


def format_few_shot_block(examples):
    blocks = []
    for ex in examples:
        block = f"Text:\n{ex['text']}\n\nLabel: {ex['label']}\nExplanation: {ex['explanation']}"
        blocks.append(block)
    return "\n\n---\n\n".join(blocks)


def parse_confidence(raw):
    for line in raw.split("\n"):
        if line.lower().startswith("confidence:"):
            val = line[len("confidence:"):].strip()
            try:
                c = int(val[0])
                if 1 <= c <= 5:
                    return c
            except:
                pass
    return 3  # default to threshold if unparseable


def annotate_with_confidence(text, model, temperature=0.0):
    few_shot_block = format_few_shot_block(FEW_SHOT_EXAMPLES)
    prompt = (
        f"{instructions}\n\n"
        f"Hier sind Beispiele:\n\n{few_shot_block}\n\n---\n\n"
        f"Jetzt annotieren Sie diesen Absatz:\n\n{text}\n\n"
        f"{reminder}\n\n"
        "Antworten Sie genau in diesem Format:\n"
        "Label: <NO_CRIME_FRAME oder CRIME_FRAME_NEG oder CRIME_FRAME_POS>\n"
        "Explanation: <ein Satz>\n"
        "Confidence: <1-5 wobei 1=sehr unsicher und 5=sehr sicher>"
    )
    messages = [
        {"role": "system", "content": "Sie sind ein Experte fuer Inhaltsanalyse. Antworten Sie immer im angegebenen Format."},
        {"role": "user",   "content": prompt}
    ]
    return call_api(messages, model, temperature)


print("Functions loaded.")

Functions loaded.


In [4]:
human_completed = pd.read_csv(RESULTS_DIR / "step3_testset_500_human_completed.csv")
human_completed = human_completed[
    human_completed["final_human_label"].isin(VALID_LABELS)
].copy().reset_index(drop=True)

print(f"Rows: {len(human_completed)}")
print(human_completed["final_human_label"].value_counts())

Rows: 250
final_human_label
NO_CRIME_FRAME     129
CRIME_FRAME_NEG    114
CRIME_FRAME_POS      7
Name: count, dtype: int64


## Step 1: Annotate with Small Model + Confidence

In [5]:
OUTPUT_SMALL = RESULTS_DIR / "confidence_pipeline_small.csv"

if OUTPUT_SMALL.exists():
    done_small = pd.read_csv(OUTPUT_SMALL)
    done_ids = set(done_small["testset_id"])
    print(f"Resuming: {len(done_small)} done")
else:
    done_small = pd.DataFrame()
    done_ids = set()
    print(f"Starting fresh: {len(human_completed)} rows")

buffer = []

for i, row in human_completed.iterrows():
    if row["testset_id"] in done_ids:
        continue

    raw        = annotate_with_confidence(str(row["text_block_en"]), MODEL_SMALL)
    label      = parse_label(raw)
    explanation= parse_explanation(raw)
    confidence = parse_confidence(raw)

    buffer.append({
        "testset_id":        row["testset_id"],
        "group":             row["group"],
        "text_block_en":     row["text_block_en"],
        "final_human_label": row["final_human_label"],
        "small_label":       label,
        "small_explanation": explanation,
        "small_confidence":  confidence,
        "needs_routing":     confidence <= CONFIDENCE_THRESHOLD,
    })
    done_ids.add(row["testset_id"])

    if len(buffer) % 50 == 0:
        chunk = pd.DataFrame(buffer)
        chunk.to_csv(OUTPUT_SMALL, mode="a", header=not OUTPUT_SMALL.exists(), index=False)
        done_small = pd.concat([done_small, chunk], ignore_index=True)
        buffer = []
        routed = done_small["needs_routing"].sum()
        print(f"  [{len(done_small)}/{len(human_completed)}] needs routing: {routed} ({routed/len(done_small)*100:.1f}%)")

if buffer:
    chunk = pd.DataFrame(buffer)
    chunk.to_csv(OUTPUT_SMALL, mode="a", header=not OUTPUT_SMALL.exists(), index=False)
    done_small = pd.concat([done_small, chunk], ignore_index=True)

print(f"\nDone. Rows needing routing: {done_small['needs_routing'].sum()}")
print(done_small["small_confidence"].value_counts().sort_index())

Starting fresh: 250 rows
  [50/250] needs routing: 49 (98.0%)
  [100/250] needs routing: 98 (98.0%)
  [150/250] needs routing: 145 (96.7%)
  [200/250] needs routing: 194 (97.0%)
  [250/250] needs routing: 244 (97.6%)

Done. Rows needing routing: 244
small_confidence
3    244
5      6
Name: count, dtype: int64


## Step 2: Route Low-Confidence Rows to Large Model

In [6]:
OUTPUT_ROUTED = RESULTS_DIR / "confidence_pipeline_routed.csv"

done_small = pd.read_csv(OUTPUT_SMALL)
to_route   = done_small[done_small["needs_routing"] == True].copy()

if OUTPUT_ROUTED.exists():
    done_routed = pd.read_csv(OUTPUT_ROUTED)
    routed_ids  = set(done_routed["testset_id"])
    print(f"Resuming routing: {len(done_routed)} done, {len(to_route) - len(done_routed)} remaining")
else:
    done_routed = pd.DataFrame()
    routed_ids  = set()
    print(f"Routing {len(to_route)} rows to {MODEL_LARGE}")

buffer = []

for i, row in to_route.iterrows():
    if row["testset_id"] in routed_ids:
        continue

    raw        = annotate_with_confidence(str(row["text_block_en"]), MODEL_LARGE)
    label      = parse_label(raw)
    explanation= parse_explanation(raw)
    confidence = parse_confidence(raw)

    buffer.append({
        "testset_id":         row["testset_id"],
        "large_label":        label,
        "large_explanation":  explanation,
        "large_confidence":   confidence,
    })
    routed_ids.add(row["testset_id"])

    if len(buffer) % 25 == 0:
        chunk = pd.DataFrame(buffer)
        chunk.to_csv(OUTPUT_ROUTED, mode="a", header=not OUTPUT_ROUTED.exists(), index=False)
        done_routed = pd.concat([done_routed, chunk], ignore_index=True)
        buffer = []
        print(f"  [{len(done_routed)}/{len(to_route)}] routed")

if buffer:
    chunk = pd.DataFrame(buffer)
    chunk.to_csv(OUTPUT_ROUTED, mode="a", header=not OUTPUT_ROUTED.exists(), index=False)
    done_routed = pd.concat([done_routed, chunk], ignore_index=True)

print(f"\nRouting done. {len(done_routed)} rows processed by {MODEL_LARGE}")

Routing 244 rows to Meta-Llama-3.3-70B-Instruct-AWQ-INT4
  [25/244] routed
  [50/244] routed
  [75/244] routed
  [100/244] routed
  [125/244] routed
  [150/244] routed
  [175/244] routed
  [200/244] routed
  [225/244] routed

Routing done. 244 rows processed by Meta-Llama-3.3-70B-Instruct-AWQ-INT4


## Step 3: Merge and Assign Final Labels

In [7]:
done_small  = pd.read_csv(OUTPUT_SMALL)
done_routed = pd.read_csv(OUTPUT_ROUTED)

merged = done_small.merge(done_routed, on="testset_id", how="left")

# final label: large model if routed, small model otherwise
merged["final_label"] = merged.apply(
    lambda r: r["large_label"] if r["needs_routing"] and pd.notna(r.get("large_label")) else r["small_label"],
    axis=1
)
merged["final_model"] = merged.apply(
    lambda r: MODEL_LARGE if r["needs_routing"] and pd.notna(r.get("large_label")) else MODEL_SMALL,
    axis=1
)

merged.to_csv(RESULTS_DIR / "confidence_pipeline_final.csv", index=False)

print(f"Total rows: {len(merged)}")
print(f"Handled by {MODEL_SMALL}: {(merged['final_model'] == MODEL_SMALL).sum()}")
print(f"Handled by {MODEL_LARGE}: {(merged['final_model'] == MODEL_LARGE).sum()}")
print(f"\nFinal label distribution:")
print(merged["final_label"].value_counts())

Total rows: 250
Handled by ministral-3-14b: 6
Handled by Meta-Llama-3.3-70B-Instruct-AWQ-INT4: 244

Final label distribution:
final_label
CRIME_FRAME_NEG    166
NO_CRIME_FRAME      82
CRIME_FRAME_POS      2
Name: count, dtype: int64


## Step 4: Evaluate

In [8]:
def evaluate(y_true, y_pred, name):
    valid = [(t, p) for t, p in zip(y_true, y_pred) if t in VALID_LABELS and p in VALID_LABELS]
    if not valid:
        print(f"{name}: no valid predictions")
        return {}
    yt, yp   = zip(*valid)
    acc      = accuracy_score(yt, yp)
    f1_macro = f1_score(yt, yp, average="macro", labels=VALID_LABELS, zero_division=0)
    f1_w     = f1_score(yt, yp, average="weighted", labels=VALID_LABELS, zero_division=0)
    kappa    = cohen_kappa_score(yt, yp, labels=VALID_LABELS)
    alpha_data = np.array([[VALID_LABELS.index(t) for t in yt], [VALID_LABELS.index(p) for p in yp]])
    try:
        alpha = krippendorff.alpha(alpha_data, level_of_measurement="nominal")
    except:
        alpha = float("nan")
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print(f"Accuracy: {acc:.3f} | Macro F1: {f1_macro:.3f} | Weighted F1: {f1_w:.3f}")
    print(f"Kappa: {kappa:.3f} | Alpha: {alpha:.3f}")
    print(classification_report(yt, yp, labels=VALID_LABELS, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(yt, yp, labels=VALID_LABELS))
    return {"name": name, "accuracy": acc, "macro_f1": f1_macro, "weighted_f1": f1_w, "kappa": kappa, "alpha": alpha}


merged = pd.read_csv(RESULTS_DIR / "confidence_pipeline_final.csv")

# confidence check: does confidence correlate with correctness?
merged["correct"] = merged["final_human_label"] == merged["final_label"]
print("Accuracy by small model confidence score:")
print(merged.groupby("small_confidence")["correct"].mean().round(3))
print()

results = evaluate(
    merged["final_human_label"].tolist(),
    merged["final_label"].tolist(),
    "Confidence Pipeline (ministral + Llama 70B routing)"
)

Accuracy by small model confidence score:
small_confidence
3    0.713
5    0.500
Name: correct, dtype: float64


Confidence Pipeline (ministral + Llama 70B routing)
Accuracy: 0.708 | Macro F1: 0.619 | Weighted F1: 0.697
Kappa: 0.447 | Alpha: 0.426
                 precision    recall  f1-score   support

 NO_CRIME_FRAME       0.85      0.54      0.66       129
CRIME_FRAME_NEG       0.63      0.92      0.75       114
CRIME_FRAME_POS       1.00      0.29      0.44         7

       accuracy                           0.71       250
      macro avg       0.83      0.58      0.62       250
   weighted avg       0.76      0.71      0.70       250

Confusion matrix:
[[ 70  59   0]
 [  9 105   0]
 [  3   2   2]]


In [9]:
# compare: small model alone vs pipeline
results_small_only = evaluate(
    merged["final_human_label"].tolist(),
    merged["small_label"].tolist(),
    "Small model only (no routing)"
)

print("\n=== Improvement from routing ===")
print(f"Kappa:       {results_small_only['kappa']:.3f} -> {results['kappa']:.3f}")
print(f"Macro F1:    {results_small_only['macro_f1']:.3f} -> {results['macro_f1']:.3f}")
print(f"Weighted F1: {results_small_only['weighted_f1']:.3f} -> {results['weighted_f1']:.3f}")


Small model only (no routing)
Accuracy: 0.796 | Macro F1: 0.618 | Weighted F1: 0.788
Kappa: 0.601 | Alpha: 0.600
                 precision    recall  f1-score   support

 NO_CRIME_FRAME       0.78      0.88      0.83       129
CRIME_FRAME_NEG       0.82      0.74      0.77       114
CRIME_FRAME_POS       1.00      0.14      0.25         7

       accuracy                           0.80       250
      macro avg       0.87      0.59      0.62       250
   weighted avg       0.80      0.80      0.79       250

Confusion matrix:
[[114  15   0]
 [ 30  84   0]
 [  2   4   1]]

=== Improvement from routing ===
Kappa:       0.601 → 0.447
Macro F1:    0.618 → 0.619
Weighted F1: 0.788 → 0.697
